In [1]:
import fastf1
import matplotlib.pyplot as plt
from fastf1 import utils
import numpy as np
import scipy as sp
import pandas as pd
import seaborn as sns

In [2]:
fastf1.Cache.clear_cache('fastf1_cache')
fastf1.Cache.enable_cache('fastf1_cache')

In [3]:
def load_session(year, gp):
    session = fastf1.get_session(year, gp, 'race')
    session.load()
    laps = session.laps
    return session

In [6]:
year = int(input("Qual o ano da corrida?"))
gp = input("Qual o GP?")
load_session(year, gp)

core           INFO 	Loading data for Japanese Grand Prix - Race [v3.0.2]
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
core        WARNING 	Failed to load extended driver information!
logger      WARNING 	Failed to load result data from Ergast!
core        WARNING 	No result data for this session available on Ergast! (This is expected for recent sessions)
logger      WARNING 	Failed to load results data!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
logger      WARNING 	Failed to load session status data!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
logger      WARNING 	Failed to load total lap count!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
logger      WARNIN

DataNotLoadedError: The data you are trying to access has not been loaded yet. See `Session.load`

In [ ]:
drivers = ['VER', 'ANT', 'LEC', "HAM", 'RUS', 'NOR', 
           'PIA','LAW', 'HUL', 'HAD', 'GAS', 'BEA', 
           'OCO', "SAI", 'LIN', 'ALO', 'PER', 'ALB',
           'BOT', 'STR', 'COL', 'BOR']  # Specify the drivers you want to analyze
car_data = {}

#specify the drivers and getting the fastest laps
for i in drivers:
    car_data[i] = laps.pick_driver(i).pick_fastest().get_car_data().add_distance()
    car_data[i]['Timestamp'] = range(len(car_data[i]))
    
    car_data[i]['Acc'] = pd.DataFrame(np.diff(car_data[i]['Speed']))
    car_data[i]['Pedal'] = pd.DataFrame(sp.integrate.cumulative_trapezoid(car_data[i]['Throttle'], car_data[i]['Timestamp'], initial=0))#Throttle integrated, it shows the trhottle trace as an straight (area under the graph)
    car_data[i]['Pedal_Acc'] = pd.DataFrame(np.diff(car_data[i]['Throttle']))


In [ ]:
car_data['HAM']

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot([car_data[i]['Speed'].dropna() for i in drivers], labels=drivers)

ax.set_ylabel('Speed')
ax.set_title('Speed per pilot (Fastest Lap)')


In [ ]:
import numpy as np

bins = np.arange(0.5, 9.5, 1)   # 0.5, 1.5, ..., 8.5

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist([car_data[i]['nGear'].dropna() for i in drivers],
        bins=bins, label=drivers)

ax.set_xticks(range(1, 9))
ax.set_xlabel('Marcha')
ax.set_ylabel('Frequency')
ax.set_title('Gear Ratio Distribution per Pilot (Fastest Lap)')
ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1.10))
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for i in drivers:
    d = car_data[i]
    idx = d['Speed'].idxmax()   # linha onde a velocidade é máxima
    ax.scatter(d.loc[idx, 'Distance'], d.loc[idx, 'Speed'],
               s=100, label=f"{i} ({d.loc[idx, 'Speed']:.0f} kph)")

ax.set_xlabel('Distance (m)')
ax.set_ylabel('Speed (kph)')
ax.set_title('Ponto de velocidade máxima por piloto')
ax.grid(alpha=0.3)
ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1.10))
plt.show()

In [ ]:
# (coluna, rótulo do eixo y)
channels = [
    ('Speed', 'Speed (kph)'),
    ('Throttle', 'Throttle (%)'),
    ('Pedal', 'Pedal Usage (-)'),
    ('Pedal_Acc', 'Pedal ACC (%/s)'),
]

fig1, axes = plt.subplots(4, figsize=(20, 10), sharex=True)

for ax, (col, ylabel) in zip(axes, channels):
    for i in drivers:
        ax.plot(car_data[i]['Distance'], car_data[i][col],
                label=i)
    ax.set_ylabel(ylabel)
    ax.grid()
    ax.legend()

axes[-1].set_xlabel('Distance (m)')
plt.tight_layout()
plt.show()


In [ ]:
point_finishers = session_data.drivers[:10]
finishing_order = [session_data.get_driver(i)["Abbreviation"] for i in point_finishers]
driver_laps = {}

for i in finishing_order:
    driver_laps[i] = laps.pick_driver(i).pick_quicklaps()
    driver_laps[i] = driver_laps[i].reset_index()

all_driver_laps = pd.concat(driver_laps.values(), ignore_index=True)

In [ ]:
finishing_order

In [ ]:
all_driver_laps.columns

In [ ]:
all_driver_laps['Sector1Time_s'] = all_driver_laps['Sector1Time'].dt.total_seconds()
all_driver_laps['Sector2Time_s'] = all_driver_laps['Sector2Time'].dt.total_seconds()
all_driver_laps['Sector3Time_s'] = all_driver_laps['Sector3Time'].dt.total_seconds()

In [ ]:
all_driver_laps['Driver'].unique()

In [ ]:
all_driver_laps['LapTime_s'] = all_driver_laps['LapTime'].dt.total_seconds()

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data = all_driver_laps, x='Driver', y='LapTime_s', ax=ax)

ax.set_ylabel('Lap Time (s)')
ax.set_xlabel('Driver')
ax.set_title('Lap Time por piloto')


###Observe os boxplots mais fechadinhos, isso significa que o piloto foi mais consistente, ou seja, teve menos variação de tempo entre as voltas.git

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data = all_driver_laps, x='Driver', y='Sector1Time_s', ax=ax)

ax.set_ylabel('Sector 1 Time (s)')
ax.set_xlabel('Driver')
ax.set_title('Sector 1 Time por piloto')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data = all_driver_laps, x='Driver', y='Sector2Time_s', ax=ax)

ax.set_ylabel('Sector 2 Time (s)')
ax.set_xlabel('Driver')
ax.set_title('Sector 2 Time por piloto')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data = all_driver_laps, x='Driver', y='Sector3Time_s', ax=ax)

ax.set_ylabel('Sector 3 Time (s)')
ax.set_xlabel('Driver')
ax.set_title('Sector 3 Time por piloto')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.scatterplot(data = all_driver_laps, y='TyreLife', x='LapTime_s', hue='Driver', ax=ax)

ax.set_xlabel('Lap Time (s)')
ax.set_ylabel('Tyre Life')
ax.set_title('Lap Time por piloto')